# Agente revisor de rendiciones de gastos — Tarea Final Curso 1 (Grupo 05)

Este notebook **solo importa** desde `src/`: la lógica vive en los módulos.
Ejecútalo de arriba abajo desde la carpeta `tarea-final-c1/`. Todos los datos son ficticios.

## 0. Ficha de reproducción

Modelo, temperatura, versiones de librerías, variables de entorno y datos usados.
La tabla completa está en `README.md`. Esta celda muestra lo que realmente está instalado.

In [ ]:
from importlib.metadata import version

from src.config import MODELO_DEL_CURSO, TEMPERATURA

print('Modelo:', MODELO_DEL_CURSO, '| temperatura:', TEMPERATURA)
for libreria in ['langgraph', 'langchain', 'langchain-google-genai', 'langchain-redis', 'redis', 'redisvl']:
    print(f'{libreria}=={version(libreria)}')

## 1. Caso y criterio de éxito

Agente revisor de rendiciones de gastos de **Comercial Los Alerces SpA** (empresa ficticia).

_TODO(caso-calidad): describir el caso y el criterio de éxito (ver README.md)._

## 2. Entorno y carga de configuración

In [ ]:
import os

from src.config import CARPETA_PROYECTO, obtener_llm, registrar_traza
from src.contrato import MAX_PASOS, DECISIONES

# src.config carga el .env del proyecto; mostramos presencia, nunca secretos.
print('Proyecto:', CARPETA_PROYECTO)
print('GEMINI_MODEL:', os.getenv('GEMINI_MODEL') or 'FALTA')
print('EMBEDDING_MODEL:', os.getenv('EMBEDDING_MODEL') or 'FALTA')
print('GOOGLE_API_KEY:', 'configurada (valor oculto)' if os.getenv('GOOGLE_API_KEY', '').strip() else 'FALTA')
print('REDIS_URL:', 'configurada (valor oculto)' if os.getenv('REDIS_URL', '').strip() else 'FALTA')
print('REDIS_PREFIJO:', os.getenv('REDIS_PREFIJO') or 'FALTA')
print('MOSTRAR_TRAZAS:', os.getenv('MOSTRAR_TRAZAS', '1'))
print('MAX_PASOS:', MAX_PASOS, '| decisiones:', ', '.join(DECISIONES))

## 3. Caso normal: ReAct con herramienta, observación, decisión y parada

La traza debe mostrar la búsqueda, su observación, la validación, su observación y la respuesta final.

In [ ]:
import csv
import json

from src.config import CARPETA_PROYECTO
from src.grafo import construir_agente, conversar

# G-001 es ficticio y sirve para recorrer el ciclo ReAct completo.
with (CARPETA_PROYECTO / 'datos' / 'rendiciones_prueba.csv').open(encoding='utf-8', newline='') as archivo:
    gastos = list(csv.DictReader(archivo))
gasto_normal = next(gasto for gasto in gastos if gasto['id'] == 'G-001')
gasto_normal['monto'] = int(gasto_normal['monto'])
gasto_normal['vb_gerente_general'] = gasto_normal['vb_gerente_general'].strip().casefold() == 'true'
for campo in ('patente', 'numero_ot'):
    gasto_normal[campo] = gasto_normal[campo] or None

agente_normal = construir_agente()
hilo_normal = 'DEMO-REACT-G001'
mensaje_normal = 'Revisa este gasto rendido: ' + json.dumps(gasto_normal, ensure_ascii=False)
respuesta_normal = conversar(agente_normal, mensaje_normal, hilo_normal)
print('Respuesta final:', respuesta_normal)
assert respuesta_normal.strip(), 'El agente debe entregar una respuesta.'

# Muestra el estado estructurado que el grafo conservó en este hilo.
estado_normal = agente_normal.get_state({'configurable': {'thread_id': hilo_normal}}).values
for campo in ('gasto', 'clasificacion', 'validacion', 'veredicto'):
    print(f'{campo}: {estado_normal.get(campo)}')

## 4. Segundo turno: uso del historial

In [ ]:
from src.grafo import construir_agente, conversar

# Un agente conserva el historial por thread_id; cada caso usa un hilo distinto.
agente_historial = construir_agente()

# GS-31: un saludo breve debe responderse sin herramientas.
respuesta_gs31 = conversar(agente_historial, 'Hola', 'GS-31')
print('GS-31:', respuesta_gs31)

# GS-32: los dos turnos usan exactamente el mismo hilo.
hilo_gs32 = 'GS-32'
respuesta_turno_1 = conversar(agente_historial, 'Soy Pedro, chofer del Centro de Distribución', hilo_gs32)
respuesta_turno_2 = conversar(agente_historial, '¿Cómo me llamo y en qué unidad trabajo?', hilo_gs32)
print('GS-32 turno 1:', respuesta_turno_1)
print('GS-32 turno 2:', respuesta_turno_2)

assert 'pedro' in respuesta_turno_2.casefold(), 'El segundo turno debe recordar el nombre.'
assert any(texto in respuesta_turno_2.casefold() for texto in ('centro de distribución', 'centro de distribucion', 'cd')), 'El segundo turno debe recordar la unidad.'
# En las trazas de ambos turnos debe verse pide_herramientas vacío.

## 5. Seguridad básica: fuera de alcance y jailbreak

In [ ]:
from src.seguridad import PROMPT_SISTEMA

# TODO(conocimiento-seguridad): pregunta fuera de alcance, jailbreak directo y jailbreak en la glosa

## 6. Ampliación RAG (Redis)

In [ ]:
from src.rag import buscar_politica

# TODO(conocimiento-seguridad): búsqueda con fragmentos y un caso donde el agente NO busca

## 7. Ampliación memoria del fondo

In [ ]:
from src.memoria import fondo_inicial, aplicar_veredicto, resumen_fondo

# TODO(herramientas): el saldo cambia y se usa en una decisión posterior

## 8. Ampliación acción: registro de veredictos

In [ ]:
from src.registro import registrar_veredicto, reiniciar_registro

# TODO(herramientas): estado antes/después y reinicio seguro

## 9. Ampliación juez

In [ ]:
from src.seguridad import juzgar

# TODO(conocimiento-seguridad): modelo, prompt y veredictos del juez visibles

## 10. Golden set: ejecución y resultados por caso

In [ ]:
import json

from src.config import CARPETA_PROYECTO

golden = json.loads((CARPETA_PROYECTO / 'pruebas' / 'golden_set.json').read_text(encoding='utf-8'))
print('Versión', golden['version'], '-', len(golden['casos']), 'casos')
# TODO(caso-calidad): ejecutar cada caso y mostrar esperado vs obtenido

## 11. Verificación final

Revisión del contrato (no necesita clave de API).

In [ ]:
import subprocess
import sys

# Usa el mismo Python del notebook (su entorno virtual).
resultado = subprocess.run([sys.executable, 'verificar.py'], capture_output=True, text=True, encoding='utf-8')
print(resultado.stdout, resultado.stderr)